# Out-of-core example of single-cell analysis on the GPU

The counts are never loaded into memory all at once. Dask reads them in blocks of cells, and
rapids-singlecell processes each block on the GPU. That's how a million cells fit on one 15 GB T4.

```
open store ─► normalize ─► highly variable genes ─► scale ─► PCA ─► (harmony) ─► neighbors ─► UMAP ─► leiden ─► save
```

Based on rapids-singlecell's [out-of-core tutorial](https://rapids-singlecell.scverse.org/en/latest/notebooks/06_out-of-core.html)
by Severin Dicks (scverse), adapted to DataScales stores.

## 1. Settings

Set `STORE` to your data asset: an AnnData `.zarr`, or a SpatialData `.zarr` (then `TABLE` picks its
table). It needs raw counts in a sparse (CSR) `X`. With `SAVE = True`, section 10 writes the results to a
new `.zarr`; the input is only read.

`RESULTS` is where `umap.png`, `cluster_summary.csv` and the default `OUTPUT` go. `OBS_COLUMNS` are
optional: those the store has are read and plotted. For an icechunk repo, uncomment the icechunk block at
the end of the settings cell.

In [ ]:
STORE = "/data/<data asset>/<name>.zarr"   # AnnData .zarr, or SpatialData .zarr
TABLE = "table"                             # SpatialData only: tables/<TABLE>
OUTPUT = None                               # None: /results/<name>.processed.zarr
SAVE = False                                # write results at the end
RESULTS = "/results"                        # the UMAP, cluster_summary.csv and the default OUTPUT
OBS_COLUMNS = ["sample_id"]                 # read and plotted if the store has them
BATCH_KEYS = []                             # for harmony, section 7
ROWS_PER_BLOCK = 24_000
repo = None

# --- icechunk (optional): uncomment this block to read a repo instead of a zarr ---
# from annizarr import Repo
# BRANCH = "main"
# ORIGIN = None          # s3://bucket/prefix behind a read-only data asset, to commit back
# repo = Repo(STORE, branch=BRANCH, origin=ORIGIN)

## 2. Start a GPU cluster

One Dask worker per GPU. With managed memory, a GPU that runs out of memory borrows host RAM
instead of failing.

In [ ]:
%matplotlib inline
import os
import shutil
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import anndata as ad
import annizarr
import cupy as cp
import dask
import numpy as np
import pandas as pd
import rapids_singlecell as rsc
import rmm
import scanpy as sc
import scipy.sparse as sp
import zarr
from anndata.experimental import read_elem_lazy
from anndata.io import read_elem, write_elem
from dask.distributed import Client
from dask_cuda import LocalCUDACluster
from rmm.allocators.cupy import rmm_cupy_allocator

dask.config.set({"distributed.scheduler.worker-ttl": None})
cluster = LocalCUDACluster(rmm_managed_memory=True, enable_cudf_spill=True, rmm_allocator_external_lib_list="cupy")
client = Client(cluster)
rmm.reinitialize(managed_memory=True)
cp.cuda.set_allocator(rmm_cupy_allocator)
client

## 3. Open the store

Nothing is written here. For a SpatialData store (no `X` at the root), its table `tables/<TABLE>` is used.

In [ ]:
if repo:
    view = Repo(ORIGIN, branch=BRANCH) if ORIGIN else repo
    SNAPSHOT = view.log()[0].id
    root = view.open_zarr("r", snapshot_id=SNAPSHOT)
else:
    root, SNAPSHOT = zarr.open_group(STORE, mode="r"), "zarr"
TABLE_PATH = "" if "X" in root else f"tables/{TABLE}"
group = root[TABLE_PATH] if TABLE_PATH else root
n_cells, n_genes = group["X"].attrs["shape"]
print(f"{n_cells:,} cells x {n_genes:,} genes")

## 4. Load the counts lazily

`X` becomes a Dask array: nothing is read yet. Of `obs`, only the columns in `OBS_COLUMNS` (those the store
has) and `BATCH_KEYS` are read, since reading every column of a large store takes a lot of memory. Empty strings
in a batch key count as missing.

In [ ]:
obs = group["obs"]
obs_columns = [c for c in OBS_COLUMNS if c in obs]
adata = ad.AnnData(
    X=read_elem_lazy(group["X"], (min(ROWS_PER_BLOCK, n_cells), n_genes)).astype(np.float32),
    obs=pd.DataFrame({c: read_elem(obs[c]) for c in dict.fromkeys(obs_columns + BATCH_KEYS)},
                     index=read_elem(obs[obs.attrs["_index"]])),
    var=read_elem(group["var"]),
)
for key in BATCH_KEYS:
    adata.obs[key] = adata.obs[key].astype("string").replace("", pd.NA).astype("category")
    if adata.obs[key].isna().any():
        warnings.warn(f"{adata.obs[key].isna().sum():,} cells have no {key}")
rsc.get.anndata_to_GPU(adata)
adata

## 5. Normalize

Scale each cell to 10,000 counts, then take the log. Still lazy: this only adds steps to the plan.
`lognorm` keeps this matrix over all genes; section 10 saves it as a layer.

In [ ]:
start = time.time()
TARGET_SUM = 1e4
rsc.pp.normalize_total(adata, target_sum=TARGET_SUM)
rsc.pp.log1p(adata)
lognorm = adata.X

## 6. Find highly variable genes

This is the first step that reads the data. Keeping only the 2,000 most variable genes makes the
matrix small enough to hold on the GPU (`persist`).

In [ ]:
%%time
rsc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
highly_variable = adata.var["highly_variable"].to_numpy()
adata = adata[:, highly_variable].copy()
adata.X = adata.X.persist()

## 7. Scale and run PCA

Scaling divides each gene by its spread, capped at 10. PCA then reduces each cell to 50 numbers.
`.compute()` brings the result out of Dask.

To correct for batches with harmony, set `BATCH_KEYS` in section 1 and uncomment the harmony lines.
Neighbors then use `X_pca_harmony`.

In [ ]:
%%time
rsc.pp.scale(adata, max_value=10, zero_center=False)
rsc.pp.pca(adata, n_comps=50, svd_solver="covariance_eigh")
adata.obsm["X_pca"] = adata.obsm["X_pca"].compute().get()
BASIS = "X_pca"

# harmony (optional): uncomment, with BATCH_KEYS set
# rsc.pp.harmony_integrate(adata, key=BATCH_KEYS, basis="X_pca", adjusted_basis="X_pca_harmony")
# adata.obsm["X_pca_harmony"] = cp.asnumpy(adata.obsm["X_pca_harmony"])
# BASIS = "X_pca_harmony"

## 8. Neighbors, UMAP and clusters

From here on the data is small, so one GPU works on it directly, without Dask.

In [ ]:
%%time
rsc.pp.neighbors(adata, n_neighbors=20, n_pcs=30, use_rep=BASIS, algorithm="ivfflat")
rsc.tl.umap(adata, min_dist=0.45, init_pos="spectral")
rsc.tl.leiden(adata, resolution=1.1)
print(f"{adata.obs['leiden'].nunique()} clusters in {time.time() - start:.0f}s")

## 9. Look at the result

The UMAP and the cells per cluster and sample also go to `RESULTS`, as `umap.png` and
`cluster_summary.csv`.

In [ ]:
Path(RESULTS).mkdir(parents=True, exist_ok=True)
fig = sc.pl.umap(adata, color=["leiden", *obs_columns], ncols=2, return_fig=True)
fig.savefig(f"{RESULTS}/umap.png", dpi=150, bbox_inches="tight")

leiden = adata.obs["leiden"]
summary = pd.crosstab(leiden, adata.obs["sample_id"]) if "sample_id" in adata.obs else pd.DataFrame(index=leiden.cat.categories)
summary.insert(0, "n_cells", leiden.value_counts())
summary.to_csv(f"{RESULTS}/cluster_summary.csv")
summary

## 10. Save the results (optional)

With `SAVE = True`, this writes `obsm/X_pca` (and `X_pca_harmony`), `obsm/X_umap`, `obs/leiden`,
`var/highly_variable`, `layers/lognorm` (with a `target_sum` attr) and `uns/process` (the settings and
the input). For a SpatialData store they go in its table. `lognorm` is computed again from the counts
and written block by block.

- **Zarr input:** a new `.zarr` at `OUTPUT` (default `<RESULTS>/<name>.processed.zarr`; `.zarr` is added
  if missing), a full copy of the input with the results added. It's built at `<OUTPUT>.partial` and renamed
  when complete. If `OUTPUT` exists, it goes to `<name>-1.zarr`, `-2`, … instead. The input is never
  changed, so a read-only data asset is fine.
- **Icechunk (the settings block uncommented):** one commit on `BRANCH`. If the branch moved since the store
  was opened, nothing is written.

In [ ]:
def child(group, key):
    if key not in group:
        write_elem(group, key, {})
    return group[key]


def put(group, key, value):
    if key in group:
        del group[key]
    write_elem(group, key, value)


def put_column(frame, name, values):
    put(frame, name, values)
    frame.attrs["column-order"] = [*(c for c in frame.attrs["column-order"] if c != name), name]


def free_path(stem, suffix=""):
    out, n = f"{stem}{suffix}", 0
    while Path(out).exists() or Path(f"{out}.partial").exists():
        n += 1
        out = f"{stem}-{n}{suffix}"
    return out


def new_output(store, output, results):
    out = output or f"{results}/{Path(store.rstrip('/')).name.removesuffix('.zarr')}.processed"
    return free_path(out.rstrip("/").removesuffix(".zarr"), ".zarr")


def save_copy(store, table, write_results, output, results):
    out = new_output(store, output, results)
    partial = f"{out}.partial"
    Path(partial).parent.mkdir(parents=True, exist_ok=True)
    if table:
        shutil.copytree(store, partial, copy_function=shutil.copyfile)
        for folder, _, _ in os.walk(partial):
            os.chmod(folder, 0o755)  # a read-only data asset's folders would be copied read-only
    else:
        annizarr.rechunk(store, output=partial)
    copy = zarr.open_group(partial, mode="r+", use_consolidated=False)
    write_results(copy[table] if table else copy)
    zarr.consolidate_metadata(partial)
    os.rename(partial, out)
    return out


def save_commit(repo, branch, snapshot, table, write_results, message, metadata=None):
    store = repo.open_zarr("w")
    if repo.log()[0].id != snapshot:
        repo.discard()
        raise RuntimeError(f"Branch {branch} moved since the store was opened; nothing was written. Reopen it and rerun.")
    try:
        write_results(store[table] if table else store)
        return repo.commit(message, metadata={"source_snapshot": snapshot, "table": table, **(metadata or {})})
    except BaseException:
        repo.discard()
        raise


def save(write_results, message):
    if repo:
        return f"Committed {save_commit(repo, BRANCH, SNAPSHOT, TABLE_PATH, write_results, message)}"
    return f"Saved to {save_copy(STORE, TABLE_PATH, write_results, OUTPUT, RESULTS)}"

In [ ]:
def write_results(group):
    host = lognorm.map_blocks(lambda b: b.get(), dtype=lognorm.dtype, meta=sp.csr_matrix((0, 0), dtype=lognorm.dtype))
    put(child(group, "layers"), "lognorm", host)
    group["layers/lognorm"].attrs["target_sum"] = TARGET_SUM
    obsm = child(group, "obsm")
    for key in ("X_pca", "X_pca_harmony", "X_umap"):
        if key in adata.obsm:
            put(obsm, key, cp.asnumpy(adata.obsm[key]).astype(np.float32))
        elif key in obsm:
            del obsm[key]
    put_column(group["obs"], "leiden", adata.obs["leiden"].values)
    put_column(group["var"], "highly_variable", highly_variable)
    put(child(group, "uns"), "process", {
        "input": STORE, "table": TABLE_PATH, "snapshot": SNAPSHOT, "basis": BASIS,
        "batch_keys": np.array(BATCH_KEYS, dtype=str), "target_sum": TARGET_SUM,
        "pca_variance_ratio": cp.asnumpy(adata.uns["pca"]["variance_ratio"]),
        "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })

In [ ]:
if SAVE:
    print(save(write_results, f"Process: {BASIS}, {leiden.nunique()} leiden clusters over {adata.n_obs:,} cells"))

## 11. Shut down

Closing the cluster frees the GPU.

In [ ]:
client.close()
cluster.close()